# Tutorial: Brainclip Colab Voice Runtime

This notebook creates an isolated virtual environment inside Colab, installs pinned dependencies, starts a FastAPI voice runtime, and exposes it through Cloudflare Tunnel.


## Why this notebook uses a virtual environment

Colab runtime packages change over time. Brainclip pins versions in a local venv so package drift in the base image does not silently break the voice server.


In [ ]:
from pathlib import Path
import os
import sys

APP_ROOT = Path('/content/brainclip_runtime')
VENV_DIR = APP_ROOT / 'venv'
APP_ROOT.mkdir(parents=True, exist_ok=True)
print('Python version:', sys.version)
if sys.version_info < (3, 10):
    raise RuntimeError('Brainclip runtime expects Python 3.10 or newer.')


In [ ]:
%%bash
set -e
mkdir -p /content/brainclip_runtime
python -m venv /content/brainclip_runtime/venv
source /content/brainclip_runtime/venv/bin/activate
python -m pip install --upgrade 'pip<25' 'setuptools<75' 'wheel<0.46'
cat > /content/brainclip_runtime/requirements.txt <<'EOF'
fastapi==0.115.8
uvicorn[standard]==0.34.0
python-multipart==0.0.20
requests==2.32.3
soundfile==0.13.1
numpy==2.1.3
scipy==1.15.2
faster-whisper==1.1.1
huggingface-hub==0.28.1
sentencepiece==0.2.0
einops==0.8.1
loguru==0.7.3
EOF
python -m pip install -r /content/brainclip_runtime/requirements.txt


In [ ]:
%%bash
set -e
cat > /content/brainclip_runtime/server.py <<'EOF'
from pathlib import Path
import io
import json
import math
import os
from typing import Any

import numpy as np
import requests
import soundfile as sf
from fastapi import FastAPI, HTTPException
from faster_whisper import WhisperModel
from pydantic import BaseModel

APP_ROOT = Path('/content/brainclip_runtime')
app = FastAPI(title='Brainclip Colab Voice Runtime')
whisper_model = None

class VoiceLine(BaseModel):
    id: str
    speaker: str
    text: str
    emotion: str = 'neutral'
    speaking_rate: float = 1.0
    pause_ms: int = 250
    temperature: float = 0.7
    chunk_length: int = 200
    normalize: bool = True

class SpeakerConfig(BaseModel):
    label: str
    modelId: str | None = None

class PresignedUrls(BaseModel):
    lines: dict[str, str]
    master: str
    transcript: str

class VoiceJobRequest(BaseModel):
    jobId: str
    userId: str
    bucket: str
    region: str = 'ap-south-1'
    lines: list[VoiceLine]
    speakerA: SpeakerConfig
    speakerB: SpeakerConfig
    presignedUrls: PresignedUrls

def get_whisper():
    global whisper_model
    if whisper_model is None:
        whisper_model = WhisperModel('large-v2', device='cuda', compute_type='float16', download_root=str(APP_ROOT / 'models'))
    return whisper_model

def fake_tts(line: VoiceLine, sample_rate: int = 44100):
    words = max(1, len(line.text.split()))
    duration = max(1.0, words / (2.4 * max(line.speaking_rate, 0.75)))
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    freq = 220 if line.speaker == 'A' else 180
    signal = 0.18 * np.sin(2 * math.pi * freq * t)
    return signal.astype(np.float32)

def wav_bytes(audio, sample_rate: int = 44100):
    buf = io.BytesIO()
    sf.write(buf, audio, sample_rate, format='WAV', subtype='PCM_16')
    return buf.getvalue()

def upload(url: str, body: bytes, content_type: str):
    response = requests.put(url, data=body, headers={'content-type': content_type}, timeout=120)
    response.raise_for_status()

def transcribe(master_path: str):
    segments, _ = get_whisper().transcribe(master_path, word_timestamps=True)
    words = []
    for segment in segments:
        for word in segment.words or []:
            words.append({'word': word.word.strip(), 'start': word.start, 'end': word.end, 'speaker': 'A'})
    return words

@app.get('/health')
def health():
    return {'status': 'ok', 'models_loaded': whisper_model is not None}

@app.post('/voice/job')
def voice_job(payload: VoiceJobRequest):
    try:
        sample_rate = 44100
        rendered = []
        combined = []
        for line in payload.lines:
            audio = fake_tts(line, sample_rate=sample_rate)
            rendered.append(audio)
            upload(payload.presignedUrls.lines[line.id], wav_bytes(audio, sample_rate), 'audio/wav')
            combined.append(audio)
            combined.append(np.zeros(int(sample_rate * (line.pause_ms / 1000)), dtype=np.float32))
        master = np.concatenate(combined) if combined else np.zeros(sample_rate, dtype=np.float32)
        master_path = APP_ROOT / 'master.wav'
        master_path.write_bytes(wav_bytes(master, sample_rate))
        upload(payload.presignedUrls.master, master_path.read_bytes(), 'audio/wav')
        words = transcribe(str(master_path))
        upload(payload.presignedUrls.transcript, json.dumps(words).encode('utf-8'), 'application/json')
        return {'jobId': payload.jobId, 'stage': 'voice_done', 'progressPct': 100}
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))
EOF


In [ ]:
%%bash
set -e
source /content/brainclip_runtime/venv/bin/activate
python -m uvicorn server:app --app-dir /content/brainclip_runtime --host 0.0.0.0 --port 8000 > /content/brainclip_runtime/uvicorn.log 2>&1 &
sleep 5
python -m pip install cloudflared >/dev/null 2>&1 || true
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
cloudflared tunnel --url http://127.0.0.1:8000 --logfile /content/brainclip_runtime/cloudflared.log > /content/brainclip_runtime/cloudflared.out 2>&1 &
sleep 8
grep -o 'https://[-a-zA-Z0-9.]*trycloudflare.com' /content/brainclip_runtime/cloudflared.out | head -n 1


## Operational notes

- Paste the printed Cloudflare URL into Brainclip settings as the Colab tunnel URL.
- Replace the placeholder TTS function with your pinned Fish Speech runtime once you confirm the exact wheel set that works with your target Colab Python image.
- The notebook already protects you from most Colab package drift by creating `/content/brainclip_runtime/venv` and reinstalling pinned versions there.
